# Lab 2 — Memory-aware domain adaptation

**Outcome:** establish a held-out baseline, adapt Parakeet with a GPU-aware trainable-layer policy, measure again, and save only the parameters changed by the lab. Estimated time: 75 minutes.

> This is a workflow exercise on a tiny public sample. WER can move in either direction and is not a production-quality result. Real claims require a representative held-out domain set.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import matplotlib.pyplot as plt
from voice_asr_lab.audio import load_dummy_librispeech, total_duration
from voice_asr_lab.asr import (
    configure_trainable_parameters, evaluate_wer, load_model_and_processor,
    save_trainable_state, tiny_finetune,
)
from voice_asr_lab.profiles import detect_profile
profile = detect_profile()
print(profile)

## 1. Preserve the evaluation boundary

The last two records stay out of training. For a customer dataset, split by speaker and recording context so near-duplicates do not leak across train and validation.

In [ ]:
records = load_dummy_librispeech(limit=12)
train_records, validation_records = records[:-2], records[-2:]
print({
    'train_examples': len(train_records),
    'validation_examples': len(validation_records),
    'train_audio_seconds': round(total_duration(train_records), 1),
    'validation_audio_seconds': round(total_duration(validation_records), 1),
})

## 2. Select trainable layers from VRAM

A100 trains the full workshop model. L4/A10 trains the CTC head and final two encoder blocks. T4 trains only the CTC head. The lighter paths demonstrate selective adaptation rather than pretending every GPU can sustain the same optimizer and activation footprint.

In [ ]:
model, processor, dtype = load_model_and_processor(training=True)
parameter_summary = configure_trainable_parameters(model, profile)
parameter_summary

In [ ]:
model.eval()
baseline = evaluate_wer(model, processor, validation_records, profile.max_audio_seconds)
print('Baseline WER:', baseline['wer'])
for reference, prediction in zip(baseline['references'], baseline['predictions']):
    print(f'REF: {reference}\nHYP: {prediction}\n')

## 3. Run the micro-fine-tune

The profile controls trainable layers, batch size, clip length, steps and learning rate. This short loop is designed to finish in a workshop and expose the mechanics—not converge the model.

In [ ]:
losses = tiny_finetune(model, processor, train_records, profile)
plt.plot(range(1, len(losses) + 1), losses, marker='o')
plt.xlabel('Optimizer step')
plt.ylabel('CTC loss')
plt.title(f'Workshop adaptation — {profile.name} profile')
plt.grid(alpha=0.25);

In [ ]:
after = evaluate_wer(model, processor, validation_records, profile.max_audio_seconds)
checkpoint = save_trainable_state(model, ROOT / 'artifacts' / 'lab2_trainable_state.pt')
print({
    'baseline_wer': baseline['wer'],
    'after_wer': after['wer'],
    'checkpoint': str(checkpoint),
    'interpretation': 'workflow smoke test only',
})

## Production checklist

Before calling an adaptation successful, define the domain slice, preserve speaker-disjoint evaluation, compare WER/CER and critical-term recall, inspect substitutions, test noise and accents, measure latency after the change, and version the data/config/checkpoint together.

**Checkpoint:** explain why your profile trained those layers and why the tiny WER movement is not a customer claim. Continue to Lab 3.